In [8]:
# Core Data Libraries 
import pandas as pd
import numpy as np

import warnings
# Keeping the notebook clean from pandas slice warnings
warnings.filterwarnings('ignore')

In [9]:
train = pd.read_csv('data/train_IPL.csv', low_memory=False)
print(f"Total rows in raw data: {train.shape[0]}")
train.head(3)
print(train['Bat First'].unique()[:15])

Total rows in raw data: 272704
['Kolkata Knight Riders' 'Chennai Super Kings' 'Rajasthan Royals'
 'Mumbai Indians' 'Deccan Chargers' 'Kings XI Punjab'
 'Royal Challengers Bangalore' 'Delhi Daredevils' 'Kochi Tuskers Kerala'
 'Pune Warriors' 'Sunrisers Hyderabad' 'Rising Pune Supergiants'
 'Gujarat Lions' 'Rising Pune Supergiant' 'Delhi Capitals']


In [10]:
#  Defining the Franchise Mapping Dictionary
team_map = {
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Delhi Capitals': 'Delhi Daredevils',
    'Punjab Kings': 'Kings XI Punjab',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Deccan Chargers': 'Sunrisers Hyderabad'
}

def clean_data(x):
    if pd.isna(x): return 'Unknown'
    return team_map.get(x, x)
cols_to_clean = ['Bat First', 'Bat Second', 'match_won_by', 'Venue']

for col in cols_to_clean:
    train[col] = train[col].fillna('Unknown').apply(clean_data)

In [11]:
print("Unique teams after cleanup:", train['Bat First'].nunique())

# The target variable depends on the final margin. I need to get max runs and wickets per innings.
def extract_stats(g):
    d = {}
    for i in [1, 2]:
        inn = g[g['Innings'] == i]
        d[f'r{i}'] = inn['Innings Runs'].max() if not inn.empty else 0
        d[f'w{i}'] = inn['Innings Wickets'].max() if not inn.empty else 0
    return pd.Series(d)

Unique teams after cleanup: 14


In [12]:
m_stats = train.groupby('Match ID').apply(extract_stats).reset_index()
df = train.groupby('Match ID').agg({
    'Bat First':'first', 
    'Bat Second':'first', 
    'Venue':'first', 
    'match_won_by':'first'
}).reset_index()

df = df.merge(m_stats, on='Match ID')
print(f"Match-level dataframe shape: {df.shape}")

Match-level dataframe shape: (1145, 9)


In [13]:
# Target Variable Logic
# A_big: Batting first wins by >20 runs
# B_big: Batting second wins by >=6 wickets
def get_target(row):
    if row['match_won_by'] == 'Unknown': return None
    
    if row['match_won_by'] == row['Bat First']:
        return 'A_big' if (row['r1'] - row['r2']) > 20 else 'A_small'
    elif row['match_won_by'] == row['Bat Second']:
        return 'B_big' if (10 - row['w2']) >= 6 else 'B_small'
    return None

In [14]:
df['target'] = df.apply(get_target, axis=1)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f"Final training matches: {len(df)}")
priors_dict = df['target'].value_counts(normalize=True).to_dict()

Final training matches: 1124


In [15]:
# Viewing the Historical Base Rate
print("Universe Anchor Base Rates:")
for key, val in priors_dict.items():
    print(f"{key}: {val:.4f}")

Universe Anchor Base Rates:
B_big: 0.3568
A_big: 0.2429
A_small: 0.2135
B_small: 0.1868


In [16]:
#  Setting up Laplace parameters
# Small sample sizes cause overconfidence therefore using alpha=10 to smooth outliers.
alpha = 10 
global_win_rate = 0.50
#  Pre-calculating global team match counts
played = pd.concat([df['Bat First'], df['Bat Second']]).value_counts()
won = df['match_won_by'].value_counts()

In [17]:
#  Laplace smoothing function
def get_smoothed_win_rate(team):
    w = won.get(team, 0)
    p = played.get(team, 0)
    
    return (w + (alpha * global_win_rate)) / (p + alpha)

df['ta_smooth_win'] = df['Bat First'].apply(get_smoothed_win_rate)
df['tb_smooth_win'] = df['Bat Second'].apply(get_smoothed_win_rate)

In [18]:
#  Defining Features and Target
features = ['Bat First', 'Bat Second', 'Venue', 'ta_smooth_win', 'tb_smooth_win']
cat_cols = ['Bat First', 'Bat Second', 'Venue']

X = df[features]
y = df['target']

In [19]:

pub_lb = pd.read_csv('data/public_lb_matches.csv')
priv_lb = pd.read_csv('data/schedule.csv')
sub = pd.read_csv('data/sample_submission.csv')
#  Pre-allocating probability arrays
pub_preds = np.zeros((len(pub_lb), 4))
priv_preds = np.zeros((len(priv_lb), 4))

In [20]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
models = []

# Training the Ensemble
print("Training highly regularized CatBoost models...")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)): 
    xtr, xvl = X.iloc[trn_idx], X.iloc[val_idx]
    ytr, yvl = y.iloc[trn_idx], y.iloc[val_idx]
    
    # Strategy: Depth=2 prevents learning noisy patterns, L2=30 punishes confident wrong answers.
    clf = CatBoostClassifier(
        iterations=150, 
        learning_rate=0.01,
        depth=2,          
        l2_leaf_reg=30,   
        loss_function='MultiClass',
        cat_features=cat_cols,
        random_seed=42 + fold,
        verbose=0
    )
    clf.fit(xtr, ytr, eval_set=(xvl, yvl))
    models.append(clf)
    classes = list(clf.classes_)

Training highly regularized CatBoost models...


In [21]:
# Forcing the final prediction to be 80% baseline. The model can only adjust by 20%.
PRIOR_W = 0.80
MODEL_W = 0.20

def get_test_feats(ta, tb, v):
    return pd.DataFrame([[ta, tb, v, get_smoothed_win_rate(ta), get_smoothed_win_rate(tb)]], columns=features)

In [22]:
#  Predicting the Public Leaderboard
for i, row in pub_lb.iterrows():
    ta, tb, v = clean_data(row['team_a']), clean_data(row['team_b']), clean_data(row['venue'])
    x_test = get_test_feats(ta, tb, v)
    
    raw_preds = np.mean([m.predict_proba(x_test)[0] for m in models], axis=0)
    p_dict = {classes[idx]: raw_preds[idx] for idx in range(4)}
    
    # Applying the 80/20 mathematical squeeze
    pub_preds[i] = [
        (p_dict['A_small'] * MODEL_W) + (priors_dict['A_small'] * PRIOR_W),
        (p_dict['A_big'] * MODEL_W) + (priors_dict['A_big'] * PRIOR_W),
        (p_dict['B_small'] * MODEL_W) + (priors_dict['B_small'] * PRIOR_W),
        (p_dict['B_big'] * MODEL_W) + (priors_dict['B_big'] * PRIOR_W)
    ]

In [23]:
# Predicting Private LB, Formatting, and Saving
# Averaging the toss scenarios since we don't know who will bat first in the future
for i, row in priv_lb.iterrows():
    h, a, v = clean_data(row['team_a']), clean_data(row['team_b']), clean_data(row['venue'])
    
    x1 = get_test_feats(h, a, v)
    x2 = get_test_feats(a, h, v)
    
    p1 = np.mean([m.predict_proba(x1)[0] for m in models], axis=0)
    p2 = np.mean([m.predict_proba(x2)[0] for m in models], axis=0)
    
    d1 = {classes[idx]: p1[idx] for idx in range(4)}
    d2 = {classes[idx]: p2[idx] for idx in range(4)}
    
    avg_a_small = 0.5 * d1['A_small'] + 0.5 * d2['B_small']
    avg_a_big = 0.5 * d1['A_big'] + 0.5 * d2['B_big']
    avg_b_small = 0.5 * d1['B_small'] + 0.5 * d2['A_small']
    avg_b_big = 0.5 * d1['B_big'] + 0.5 * d2['A_big']
    
    priv_preds[i] = [
        (avg_a_small * MODEL_W) + (priors_dict['A_small'] * PRIOR_W),
        (avg_a_big * MODEL_W) + (priors_dict['A_big'] * PRIOR_W),
        (avg_b_small * MODEL_W) + (priors_dict['B_small'] * PRIOR_W),
        (avg_b_big * MODEL_W) + (priors_dict['B_big'] * PRIOR_W)
    ]

results = []
for i, row in pub_lb.iterrows(): results.append([str(row['match_id'])] + list(pub_preds[i]))
for i, row in priv_lb.iterrows(): results.append([str(row['match_id'])] + list(priv_preds[i]))

final_df = pd.DataFrame(results, columns=['match_id', 'A_small', 'A_big', 'B_small', 'B_big'])
sub_out = sub[['match_id']].astype(str).merge(final_df, on='match_id', how='left').fillna(0.25)
sub_out.to_csv('submission_mark6.csv', index=False)

print("Successfully generated submission_mark6.csv")

Successfully generated submission_mark6.csv
